# NB_MDP_20 · Silver (una entidad)

Lo llama `NB_MDP_00_orquestador` después del bronze de la misma entidad.

1. Lee de bronze **solo las filas de la corrida** (`_run_id`). Para reprocesar una corrida vieja, pasá su id en `bronze_run_id`.
2. Si la entidad tiene columnas en `meta.ENTITY_COLUMN`, selecciona, renombra, transforma y tipa. Si no tiene, pasa todas las columnas.
3. Aplica las reglas de `meta.DQ_RULE`: `WARN` cuenta, `DROP` manda la fila a `<esquema silver>.dq_rechazos` (sin columnas PII) y `FAIL` corta la entidad.
4. Deduplica por `PRIMARY_KEY`, quedándose con la versión más nueva (`ORDER_COLUMN`, o `_ingested_at`).
5. Escribe con `MERGE` (upsert por clave), `OVERWRITE` o `APPEND`, según `SILVER_MODE`.

Como el `MERGE` es por clave, correrlo dos veces con los mismos datos da el mismo resultado.

In [ ]:
# Parámetros
run_id = "manual"
entity_json = "{}"
bronze_run_id = ""      # vacío = la corrida actual

In [ ]:
import json
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

e = json.loads(entity_json)
resultado = {"layer": "silver", "object_code": e.get("entity_code"), "status": None,
             "rows_read": None, "rows_written": None, "rows_rejected": None,
             "dq_warnings": None, "error": None}

AUDITORIA = ["_run_id", "_ingested_at"]


def tabla_rechazos(silver_table):
    """silver.clientes -> silver.dq_rechazos ; LH.silver.clientes -> LH.silver.dq_rechazos"""
    partes = silver_table.split(".")
    return ".".join(partes[:-1] + ["dq_rechazos"])


def aplicar_columnas(df, columnas):
    if not columnas:
        return df
    exprs = [F.expr(c.get("transform_expr") or f"`{c['source_column']}`")
               .cast(c["target_type"]).alias(c["target_column"]) for c in columnas]
    return df.select(*exprs, *AUDITORIA)


def aplicar_calidad(df, reglas, columnas_pii):
    """Devuelve (df_ok, df_rechazos o None, avisos). Una regla que da NULL cuenta como incumplida."""
    avisos, rechazos = {}, None
    for r in reglas or []:
        cumple = F.coalesce(F.expr(r["rule_expr"]), F.lit(False))
        if r["action"] == "FAIL":
            n = df.where(~cumple).count()
            if n:
                raise ValueError(f"Regla {r['rule_name']} (FAIL): {n} filas no cumplen {r['rule_expr']}")
        elif r["action"] == "WARN":
            n = df.where(~cumple).count()
            if n:
                avisos[r["rule_name"]] = n
        elif r["action"] == "DROP":
            malas = df.where(~cumple).withColumn("_regla", F.lit(r["rule_name"]))
            rechazos = malas if rechazos is None else rechazos.unionByName(malas)
            df = df.where(cumple)
    if rechazos is not None:
        # Las columnas PII no se copian a la tabla de rechazos
        visibles = [c for c in rechazos.columns if c not in columnas_pii and c not in ("_regla", *AUDITORIA)]
        rechazos = rechazos.select(
            F.lit(e["entity_code"]).alias("entity_code"),
            F.col("_regla").alias("rule_name"),
            F.col("_run_id").alias("run_id"),
            F.to_json(F.struct(*visibles)).alias("row_json"),
            F.current_timestamp().alias("rejected_at"))
    return df, rechazos, avisos


def deduplicar(df, pk, order_column):
    if not pk:
        return df
    orden = [F.col(order_column).desc_nulls_last()] if order_column else []
    w = Window.partitionBy(*pk).orderBy(*orden, F.col("_ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")


def filas_escritas(tabla):
    m = DeltaTable.forName(spark, tabla).history(1).select("operationMetrics").first()[0] or {}
    if "numTargetRowsInserted" in m:   # MERGE: numOutputRows incluye filas reescritas sin cambios
        return int(m["numTargetRowsInserted"]) + int(m.get("numTargetRowsUpdated", 0))
    return int(m.get("numOutputRows", 0))

In [ ]:
try:
    origen_run = bronze_run_id or run_id
    if not spark.catalog.tableExists(e["bronze_table"]):
        resultado["status"] = "SKIPPED_NO_DATA"
    else:
        df = spark.table(e["bronze_table"]).where(F.col("_run_id") == origen_run)
        df = aplicar_columnas(df, e.get("columns")).persist()
        leidas = df.count()
        if leidas == 0:
            resultado.update(status="SKIPPED_NO_DATA", rows_read=0, rows_written=0)
        else:
            pii = [c["target_column"] for c in (e.get("columns") or []) if c.get("is_pii")]
            df_ok, rechazos, avisos = aplicar_calidad(df, e.get("dq_rules"), pii)

            if rechazos is not None:
                rechazos = rechazos.persist()
                n_rech = rechazos.count()
                if n_rech:
                    rechazos.write.format("delta").mode("append").saveAsTable(tabla_rechazos(e["silver_table"]))
                rechazos.unpersist()
            else:
                n_rech = 0

            pk = [k.strip() for k in (e.get("primary_key") or "").split(",") if k.strip()]
            df_ok = deduplicar(df_ok, pk, e.get("order_column"))
            df_ok = df_ok.withColumn("_silver_updated_at", F.current_timestamp())

            destino, modo = e["silver_table"], e["silver_mode"]
            if modo == "OVERWRITE" or not spark.catalog.tableExists(destino):
                (df_ok.write.format("delta").mode("overwrite")
                      .option("overwriteSchema", "true").saveAsTable(destino))
            elif modo == "APPEND":
                df_ok.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(destino)
            else:  # MERGE: upsert por clave
                cond = " AND ".join(f"t.`{k}` <=> s.`{k}`" for k in pk)
                spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
                (DeltaTable.forName(spark, destino).alias("t")
                    .merge(df_ok.alias("s"), cond)
                    .whenMatchedUpdateAll()
                    .whenNotMatchedInsertAll()
                    .execute())
            escritas = filas_escritas(destino)   # antes del ALTER, que también queda en el historial

            if pii:
                # Deja marcadas las columnas PII en la tabla, para gobierno y para la skill de documentación
                spark.sql(f"ALTER TABLE {destino} SET TBLPROPERTIES ('mdp.pii_columns' = '{','.join(pii)}')")

            resultado.update(status="SUCCEEDED", rows_read=leidas, rows_written=escritas,
                             rows_rejected=n_rech, dq_warnings=avisos or None)
        df.unpersist()
except Exception as ex:
    resultado.update(status="FAILED", error=f"{type(ex).__name__}: {ex}"[:3000])

print(json.dumps(resultado, ensure_ascii=False, indent=2, default=str))

In [ ]:
notebookutils.notebook.exit(json.dumps(resultado, ensure_ascii=False, default=str))